In [ ]:
# Install all required packages and mounts Google Drive
!pip install rasterio geopandas shapely numpy scikit-learn kneed \
             matplotlib fiona pyproj rtree scipy -q

from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/pipeline"

for folder in [
    f"{BASE}/src",
    f"{BASE}/analysis",
    f"{BASE}/data/raw",
    f"{BASE}/data/plots",
    f"{BASE}/data/results",
]:
    os.makedirs(folder, exist_ok=True)

open(f"{BASE}/src/__init__.py", "a").close()
open(f"{BASE}/analysis/__init__.py", "a").close()

print("Setup complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete.


In [ ]:
# config.py
# Central configuration
%%writefile /content/pipeline/config.py
from pathlib import Path

# Root directory of the pipeline
BASE_DIR = Path("/content/pipeline")

# Input file paths on Google Drive
SOURCE_IMAGE  = Path("ADD_PATH")
BOUNDARY_FILE = Path("ADD_PATH")

# Output directories
RAW_DIR     = BASE_DIR / "data/raw"
PLOT_DIR    = BASE_DIR / "data/plots"
RESULTS_DIR = BASE_DIR / "data/results"

# Band order string
BAND_ORDER = "BGRNIR"

# Kmeans search range and pixel sampling limits
K_MIN       = 2
K_MAX       = 13
SAMPLE_SIZE = 100_000
SIL_SAMPLE  = 5_000
BATCH_SIZE  = 20_000

# Raster and vector processing settings
NODATA_LABEL = 255
CONNECTIVITY = 4
CHUNK_SIZE   = 500_000
SAVE_SHP     = False

Overwriting /content/pipeline/config.py


In [ ]:
# utils.py
# shared helper functions used across all pipeline stages
%%writefile /content/pipeline/utils.py
import time
import csv
import json
import logging
import numpy as np
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
logging.getLogger("rasterio").setLevel(logging.ERROR)

#  palette
PASTEL = {
    "vegetation":     "#A8D5B5",
    "non_vegetation": "#F5C89A",
    "water_road":     "#A3C4E8",
    "accent":         "#C4A8D5",
    "neutral":        "#D4D0C8",
    "bg":             "#FAFAF8",
    "grid":           "#EDEBE5",
    "text":           "#3A3A38",
    "text_muted":     "#7A7A76",
    "blue":           "#A3C4E8",
    "green":          "#A8D5B5",
    "orange":         "#F5C89A",
    "purple":         "#C4A8D5",
}

# Shared matplotlib style
def _apply_base_style(ax, grid_axis="y"):
    ax.set_facecolor(PASTEL["bg"])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(PASTEL["grid"])
    ax.spines["bottom"].set_color(PASTEL["grid"])
    ax.tick_params(colors=PASTEL["text_muted"], labelsize=9)
    ax.xaxis.label.set_color(PASTEL["text_muted"])
    ax.yaxis.label.set_color(PASTEL["text_muted"])
    if grid_axis in ("y", "both"):
        ax.yaxis.grid(True, color=PASTEL["grid"], linewidth=0.8, zorder=0)
    if grid_axis in ("x", "both"):
        ax.xaxis.grid(True, color=PASTEL["grid"], linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

def fmt_time(seconds: float) -> str:
    m, s = divmod(int(seconds), 60)
    return f"{m}m {s:02d}s"

class Timer:
    def __enter__(self):
        self._start = time.perf_counter()
        return self

    def __exit__(self, *_):
        self.elapsed = time.perf_counter() - self._start

    def pretty(self) -> str:
        return fmt_time(self.elapsed)

def log(msg: str):
    print(f"[INFO] {msg}")

def log_table(rows: list, headers: list, col_widths: list):
    fmt = "  " + "  ".join(f"{{:<{w}}}" for w in col_widths)
    sep = "  " + "  ".join("-" * w for w in col_widths)
    print(fmt.format(*headers))
    print(sep)
    for row in rows:
        print(fmt.format(*[str(v) for v in row]))

def is_band_valid(band: np.ndarray, threshold: float = 1.0) -> bool:
    return float(np.std(band)) > threshold

# Elbow / silhouette plot
def save_elbow_plot(ks, wcss, sil, best_k, stem, plot_dir):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5),
                                    facecolor="white", constrained_layout=True)

    for ax, y, ylabel, color in [
        (ax1, wcss, "WCSS (inertia)", PASTEL["blue"]),
        (ax2, sil,  "Silhouette score", PASTEL["green"]),
    ]:
        _apply_base_style(ax, grid_axis="y")
        ax.fill_between(ks, y, alpha=0.18, color=color, zorder=1)
        ax.plot(ks, y, "-o", color=color, linewidth=2, markersize=6,
                markerfacecolor="white", markeredgewidth=1.8,
                markeredgecolor=color, zorder=3)
        ki = ks.index(best_k)
        ax.plot(best_k, y[ki], "o", color=PASTEL["accent"],
                markersize=10, zorder=4, markeredgewidth=0)
        ax.axvline(best_k, color=PASTEL["accent"], linewidth=1.4,
                   linestyle="--", alpha=0.7, zorder=2,
                   label=f"selected k = {best_k}")
        ax.set_xlabel("number of clusters (k)", fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
        ax.legend(fontsize=9, frameon=False,
                  labelcolor=PASTEL["text_muted"])

    fig.suptitle(f"optimal k selection — {stem}",
                 fontsize=12, color=PASTEL["text"], y=1.02)
    plt.savefig(Path(plot_dir) / f"{stem}_k_selection.png",
                dpi=180, bbox_inches="tight", facecolor="white")
    plt.close()

# k-scores CSV
def save_k_scores_csv(ks, wcss, sil, best_k, stem, plot_dir):
    out = Path(plot_dir) / f"{stem}_k_scores.csv"
    with open(out, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["k", "wcss", "silhouette", "selected"])
        for k, w, s in zip(ks, wcss, sil):
            writer.writerow([k, round(w, 4), round(s, 6),
                             "YES" if k == best_k else ""])

def save_json(data: dict, path: Path):
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)

Writing /content/pipeline/utils.py


In [ ]:
# src/kmeans_classification.py
# Stage 1 - feature engineering, PCA, MiniBatchKMeans, elbow/silhouette search
%%writefile /content/pipeline/src/kmeans_classification.py
import numpy as np
import rasterio
from rasterio.enums import ColorInterp
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from kneed import KneeLocator
import sys
sys.path.insert(0, "/content/pipeline")
from config import (
    NODATA_LABEL, K_MIN, K_MAX, SAMPLE_SIZE,
    SIL_SAMPLE, BATCH_SIZE, BAND_ORDER,
)
from utils import Timer, save_elbow_plot, save_k_scores_csv, is_band_valid, log

def _detect_bands(image, n_bands, desc, ci, band_order):
    def _find(keys, target):
        for i, d in enumerate(desc):
            if d and any(k in d.lower() for k in keys):
                return i
        for i, c in enumerate(ci):
            if c == target:
                return i
        return None

    r   = _find(["red"],             ColorInterp.red)
    g   = _find(["green"],           ColorInterp.green)
    b   = _find(["blue"],            ColorInterp.blue)
    nir = _find(["nir", "infrared"], ColorInterp.undefined)

    if None in (r, g, b):
        order   = band_order.upper().replace(" ", "")
        mapping = {}
        for idx, ch in enumerate(order):
            if ch not in mapping:
                mapping[ch] = idx
        r   = mapping.get("R")
        g   = mapping.get("G")
        b   = mapping.get("B")
        nir = mapping.get("N")

    if nir is not None and nir < n_bands:
        nir = nir if is_band_valid(image[nir]) else None
    else:
        nir = None
    return r, g, b, nir

def _build_features(image, r, g, b, nir):
    from scipy.ndimage import uniform_filter
    eps = 1e-6
    features, names = [], []

    for i in range(image.shape[0]):
        features.append(image[i].ravel())
        names.append(f"Band_{i + 1}")

    features.append(image.mean(axis=0).ravel())
    names.append("Mean_Brightness")

    R   = image[r].ravel()   if r   is not None else None
    G   = image[g].ravel()   if g   is not None else None
    B   = image[b].ravel()   if b   is not None else None
    NIR = image[nir].ravel() if nir is not None else None
    R2D = image[r]           if r   is not None else None
    G2D = image[g]           if g   is not None else None
    B2D = image[b]           if b   is not None else None

    if all(v is not None for v in (R, G, B)):
        features.append(2 * G - R - B);                       names.append("ExG")
        features.append((G - R) / (G + R - B + eps));         names.append("VARI")
        features.append((2*G - R - B) / (2*G + R + B + eps)); names.append("GLI")
        features.append(G / (R + G + B + eps));                names.append("GreenRatio")

    if all(v is not None for v in (R, B)):
        features.append(B / (R + eps)); names.append("BR")

    if all(v is not None for v in (R, G)):
        features.append((G - R) / (G + R + eps)); names.append("NGRDI")
        features.append(1.4 * R - G);             names.append("ExR")

    if all(v is not None for v in (R2D, G2D, B2D)):
        maxc = np.maximum(np.maximum(R2D, G2D), B2D)
        diff = maxc - np.minimum(np.minimum(R2D, G2D), B2D) + eps
        hue  = np.zeros_like(maxc)
        mr, mg, mb = maxc == R2D, maxc == G2D, maxc == B2D
        hue[mr] = ((G2D[mr] - B2D[mr]) / diff[mr]) % 6
        hue[mg] = (B2D[mg] - R2D[mg]) / diff[mg] + 2
        hue[mb] = (R2D[mb] - G2D[mb]) / diff[mb] + 4
        features.append((hue / 6.0).ravel()); names.append("Hue")

    if G2D is not None:
        gn = G2D / (G2D.max() + eps)
        features.append(np.sqrt(np.maximum(
            uniform_filter(gn**2, 5) - uniform_filter(gn, 5)**2, 0
        )).ravel()); names.append("Texture_G")

    if R2D is not None:
        rn = R2D / (R2D.max() + eps)
        features.append(np.sqrt(np.maximum(
            uniform_filter(rn**2, 5) - uniform_filter(rn, 5)**2, 0
        )).ravel()); names.append("Texture_R")

    if all(v is not None for v in (NIR, R)):
        features.append((NIR - R) / (NIR + R + eps));               names.append("NDVI")
        features.append(((NIR - R) / (NIR + R + 0.5 + eps)) * 1.5); names.append("SAVI")
        if G is not None:
            features.append((G - NIR) / (G + NIR + eps));           names.append("MNDWI")

    X = np.column_stack(features)
    col_means = np.nanmean(X, axis=0)
    bad = ~np.isfinite(X)
    X[bad] = np.take(col_means, np.where(bad)[1])
    return X, names

def _classify_one(img_path, output_dir, plot_dir, band_order):
    # run the full classification pipeline for a single image
    out_path   = output_dir / (img_path.stem + "_classified.tif")
    plot_path  = plot_dir   / f"{img_path.stem}_k_selection.png"
    csv_path   = plot_dir   / f"{img_path.stem}_k_scores.csv"

    if out_path.exists() and plot_path.exists() and csv_path.exists():
        import csv as _csv
        with open(str(csv_path)) as f:
            for row in _csv.DictReader(f):
                if row["selected"] == "YES":
                    return int(row["k"]), None, None, None, None
        return None, None, None, None, None

    with rasterio.open(str(img_path)) as src:
        image   = src.read().astype(np.float32)
        profile = src.profile
        desc    = src.descriptions or [None] * src.count
        ci      = src.colorinterp
        nodata  = src.nodata

    bands, height, width = image.shape

    valid_mask = (
        ~np.any(image == nodata, axis=0).ravel()
        if nodata is not None
        else np.ones(height * width, dtype=bool)
    )

    r, g, b, nir = _detect_bands(image, bands, desc, ci, band_order)

    band_labels = []
    for label, idx in [("B", b), ("G", g), ("R", r)]:
        if idx is not None:
            band_labels.append(f"{label}={idx + 1}")
    band_labels.append(f"NIR={nir + 1}" if nir is not None else "NIR=auto-dead")

    X_all, feature_names = _build_features(image, r, g, b, nir)
    X_valid = StandardScaler().fit_transform(X_all[valid_mask])
    rng     = np.random.default_rng(42)
    fit_idx = rng.choice(len(X_valid), min(SAMPLE_SIZE, len(X_valid)), replace=False)
    pca     = PCA(n_components=0.99, random_state=42).fit(X_valid[fit_idx])
    X_pca   = pca.transform(X_valid)

    search_idx = rng.choice(len(X_pca), min(SAMPLE_SIZE, len(X_pca)), replace=False)
    search     = X_pca[search_idx]
    sil_sub    = search[rng.choice(len(search), min(SIL_SAMPLE, len(search)), replace=False)]

    ks, wcss, sil = list(range(K_MIN, K_MAX + 1)), [], []
    for k in ks:
        km = MiniBatchKMeans(n_clusters=k, batch_size=BATCH_SIZE, random_state=42)
        km.fit(search)
        wcss.append(km.inertia_)
        sil.append(silhouette_score(sil_sub, km.predict(sil_sub)))

    kneedle = KneeLocator(ks, wcss, curve="convex", direction="decreasing", S=0.5)
    best_k  = kneedle.knee or ks[0]

    save_elbow_plot(ks, wcss, sil, best_k, img_path.stem, plot_dir)
    save_k_scores_csv(ks, wcss, sil, best_k, img_path.stem, plot_dir)

    km_final = MiniBatchKMeans(
        n_clusters=best_k, batch_size=BATCH_SIZE, random_state=42
    ).fit(search)
    labels = km_final.predict(X_pca)

    full = np.full(height * width, NODATA_LABEL, dtype=np.uint8)
    full[valid_mask] = labels.astype(np.uint8)
    profile.update(count=1, dtype=rasterio.uint8, nodata=NODATA_LABEL)
    with rasterio.open(str(out_path), "w", **profile) as dst:
        dst.write(full.reshape(height, width), 1)

    band_str = f"{bands} bands [{', '.join(band_labels)}]"
    return best_k, wcss[ks.index(best_k)], sil[ks.index(best_k)], band_str, feature_names


def run_classification(raw_dir, results_dir, plot_dir, image_configs):
    # entry point for stage 1 - processes all tif/tiff files in raw_dir
    results_dir.mkdir(parents=True, exist_ok=True)
    images = sorted(raw_dir.glob("*.tif")) + sorted(raw_dir.glob("*.tiff"))
    if not images:
        log(f"No .tif images found in {raw_dir}")
        return None

    log("Stage 1: KMeans Classification")
    print()

    best_k_out = None
    for img_path in images:
        cfg        = image_configs.get(img_path.name, {})
        band_order = cfg.get("band_order", BAND_ORDER)

        log(f"Processing: {img_path.name}")
        with Timer() as t:
            best_k, wcss_val, sil_val, band_info, feats = _classify_one(
                img_path, results_dir, plot_dir, band_order
            )

        if best_k is not None:
            best_k_out = best_k

        if wcss_val is not None:
            print(f"  Bands      : {band_info}")
            print(f"  Features   : {len(feats)}")
            print(f"  best_k = {best_k} | WCSS = {wcss_val:,.2f} | Sil = {sil_val:.4f}")
            print(f"  Output     : {img_path.stem}_classified.tif")
            print(f"Stage 1 completed in {t.pretty()}")
        else:
            print(f"  Skipped - already classified  (k = {best_k})")
        print()

    return best_k_out

Writing /content/pipeline/src/kmeans_classification.py


In [ ]:
# src/vectorize_raster.py
# Stage 2 - converts classified raster to polygon GeoPackage
%%writefile /content/pipeline/src/vectorize_raster.py
import os
import gc
import geopandas as gpd
import rasterio
from rasterio.features import shapes
from shapely.geometry import shape
import sys
sys.path.insert(0, "/content/pipeline")
from config import CONNECTIVITY, NODATA_LABEL
from utils import Timer, log

def _vectorize_one(raster_path, output_dir):
    # convert a single classified raster to polygon gpkg
    out_path = output_dir / (raster_path.stem.replace("_classified", "_vector") + ".gpkg")
    if out_path.exists():
        return out_path, None

    with rasterio.open(str(raster_path)) as src:
        raster    = src.read(1)
        transform = src.transform
        crs       = src.crs

    records = [
        {"geometry": shape(geom), "DN": int(val)}
        for geom, val in shapes(raster, transform=transform, connectivity=CONNECTIVITY)
    ]
    gdf = gpd.GeoDataFrame(records, crs=crs)

    # remove nodata polygons before saving
    gdf = gdf[gdf["DN"] != NODATA_LABEL].reset_index(drop=True)
    gdf.insert(0, "FID", range(1, len(gdf) + 1))
    gdf = gdf[["FID", "DN", "geometry"]]
    gdf.to_file(str(out_path), driver="GPKG")

    n = len(gdf)
    del records, gdf
    gc.collect()
    return out_path, n

def run_vectorization(results_dir):
    # entry point for stage 2 - vectorises all classified rasters in results_dir
    rasters = sorted(f for f in os.listdir(results_dir) if f.endswith("_classified.tif"))
    if not rasters:
        log("No classified rasters found")
        return

    log("Stage 2: Raster to Vector Conversion")
    print()

    for r in rasters:
        log(f"Processing: {r}")
        with Timer() as t:
            out_path, n_polys = _vectorize_one(results_dir / r, results_dir)

        if n_polys is not None:
            print(f"  Polygons : {n_polys:,}")
            print(f"  Output   : {out_path.name}")
            print(f"Stage 2 completed in {t.pretty()}")
        else:
            print(f"  Skipped - already vectorised")
        print()
        gc.collect()

Writing /content/pipeline/src/vectorize_raster.py


In [ ]:
# src/class_mapping.py
# Stage 3 - spatially joins land cover polygons to cadastral parcels
%%writefile /content/pipeline/src/class_mapping.py
import os
import gc
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import shapely
from shapely import STRtree
from shapely.geometry import box
from pathlib import Path
import sys
sys.path.insert(0, "/content/pipeline")
from config import SAVE_SHP, CHUNK_SIZE
from utils import Timer, log

def _load_boundary(boundary_path):
    # load boundary from folder containing shp or directly from shp/gpkg file
    p = Path(boundary_path)
    if p.is_dir():
        shps = list(p.glob("*.shp"))
        if not shps:
            raise FileNotFoundError(f"No .shp in: {p}")
        p = shps[0]
        log(f"Boundary folder detected - using: {p.name}")
    if not p.exists():
        raise FileNotFoundError(f"Not found: {p}")
    if p.suffix.lower() not in (".gpkg", ".shp"):
        raise ValueError(f"Unsupported format: {p.suffix}")
    return gpd.read_file(str(p))

def _raster_bbox(raster_path, target_crs):
    # return raster bounding box as shapely polygon reprojected to target_crs
    import pyproj
    from shapely.ops import transform as shp_transform

    with rasterio.open(str(raster_path)) as src:
        bounds     = src.bounds
        raster_crs = src.crs

    bbox = box(bounds.left, bounds.bottom, bounds.right, bounds.top)

    if raster_crs.to_epsg() != target_crs.to_epsg():
        project = pyproj.Transformer.from_crs(
            raster_crs, target_crs, always_xy=True
        ).transform
        bbox = shp_transform(project, bbox)
    return bbox

def _save(gdf, path, save_shp):
    # save geodataframe to gpkg and optionally to shapefile
    gdf.reset_index(drop=True).to_file(str(path), driver="GPKG")
    if save_shp:
        gdf.to_file(str(path.with_suffix(".shp")))

def _map_one(vector_file, cadastral, output_dir, raster_path=None):
    # assign dominant land cover cluster to each cadastral parcel
    stem       = vector_file.stem.replace("_vector", "")
    inter_path = output_dir / f"{stem}_intersections.gpkg"
    final_path = output_dir / f"{stem}_final.gpkg"

    if inter_path.exists() and final_path.exists():
        return None

    lc  = gpd.read_file(str(vector_file))
    utm = cadastral.estimate_utm_crs()
    ca  = cadastral.to_crs(utm).reset_index(drop=True)
    lc  = lc.to_crs(utm).reset_index(drop=True)

    ca["parcel_fid"] = range(1, len(ca) + 1)
    ca["parcel_m2"]  = ca.geometry.area.astype(np.float64)
    ca = ca[ca["parcel_m2"] >= 1.0].reset_index(drop=True)

    if raster_path is not None:
        bbox         = _raster_bbox(raster_path, utm)
        inside_mask  = ca.geometry.intersects(bbox)
        n_outside    = int((~inside_mask).sum())
        ca_inside    = ca[inside_mask].reset_index(drop=True)
        ca_outside   = ca[~inside_mask].reset_index(drop=True)
        print(f"  Parcels in boundary      : {len(ca):,}")
        print(f"  Outside raster extent    : {n_outside:,}  (labelled DN=-1)")
        print(f"  Parcels inside raster    : {len(ca_inside):,}")
    else:
        ca_inside  = ca.copy()
        ca_outside = gpd.GeoDataFrame(columns=ca.columns, crs=utm)
        print(f"  Total parcels            : {len(ca_inside):,}")

    if ca_inside.empty:
        log("WARNING: no parcels inside raster extent")
        return {"parcels": 0, "features": 0}

    candidates = list(ca_inside.sindex.intersection(lc.total_bounds))
    ca_clipped = ca_inside.iloc[candidates].copy().reset_index(drop=True)

    if ca_clipped.empty:
        log("WARNING: no overlap between image and boundary")
        return {"parcels": 0, "features": 0}

    lc_geoms = lc.geometry.values
    ca_geoms = ca_clipped.geometry.values
    fid_arr  = ca_clipped["parcel_fid"].values

    print("  Running spatial join...")
    tree = STRtree(lc_geoms)
    ca_idxs, lc_idxs = tree.query(ca_geoms, predicate="intersects")
    print(f"  Candidate pairs          : {len(ca_idxs):,}")

    if len(ca_idxs) == 0:
        log("WARNING: no intersecting pairs found")
        return {"parcels": len(ca_clipped), "features": 0}

    lc_matched  = lc_geoms[lc_idxs]
    ca_matched  = ca_geoms[ca_idxs]
    dn_matched  = lc["DN"].values[lc_idxs]
    fid_matched = fid_arr[ca_idxs]

    n_chunks = max(1, len(ca_idxs) // CHUNK_SIZE + 1)
    print(f"  Chunks                   : {n_chunks} x {CHUNK_SIZE:,}")

    all_geom, all_area, all_dn, all_fid = [], [], [], []

    for i in range(0, len(ca_idxs), CHUNK_SIZE):
        sl    = slice(i, i + CHUNK_SIZE)
        inter = shapely.intersection(ca_matched[sl], lc_matched[sl])
        area  = shapely.area(inter)
        mask  = area > 0.5
        if not mask.any():
            continue
        all_geom.extend(inter[mask])
        all_area.extend(area[mask])
        all_dn.extend(dn_matched[sl][mask])
        all_fid.extend(fid_matched[sl][mask])
    gc.collect()

    if not all_fid:
        log("No valid intersections found")
        return {"parcels": len(ca_clipped), "features": 0}

    fid_np  = np.array(all_fid,  dtype=np.int64)
    dn_np   = np.array(all_dn,   dtype=np.int64)
    area_np = np.round(np.array(all_area, dtype=np.float64), 2)

    inter_df     = pd.DataFrame({"parcel_fid": fid_np, "DN": dn_np, "area_m2": area_np})
    parcel_total = inter_df.groupby("parcel_fid")["area_m2"].transform("sum")
    pct_np       = np.round((area_np / parcel_total.values) * 100, 2)

    inter_gdf = gpd.GeoDataFrame(
        {"parcel_fid": fid_np, "DN": dn_np,
         "area_m2": area_np, "percentage": pct_np},
        geometry=all_geom, crs=utm,
    )[["parcel_fid", "DN", "area_m2", "percentage", "geometry"]]
    _save(inter_gdf, inter_path, SAVE_SHP)

    grp = inter_df.groupby(["parcel_fid", "DN"],
                            sort=False)["area_m2"].sum().reset_index()
    dominant = (
        grp.loc[grp.groupby("parcel_fid")["area_m2"].idxmax()]
        .reset_index(drop=True)
        .rename(columns={"area_m2": "dominant_area_m2"})
    )

    all_inside = pd.DataFrame({
        "parcel_fid": ca_inside["parcel_fid"].values,
        "parcel_m2":  ca_inside["parcel_m2"].values,
        "geometry":   ca_inside.geometry.values,
    })
    final = all_inside.merge(dominant, on="parcel_fid", how="left")
    final["percentage"] = (
        (final["dominant_area_m2"] / final["parcel_m2"]) * 100
    ).round(2)

    inside_gdf = gpd.GeoDataFrame(
        {
            "parcel_fid": final["parcel_fid"].astype(int),
            "DN":         final["DN"].fillna(-1).astype(int),
            "area_m2":    final["dominant_area_m2"].fillna(0).round(2),
            "percentage": final["percentage"].fillna(0),
        },
        geometry=final["geometry"].values, crs=utm,
    )[["parcel_fid", "DN", "area_m2", "percentage", "geometry"]]

    if not ca_outside.empty:
        outside_gdf = gpd.GeoDataFrame(
            {
                "parcel_fid": ca_outside["parcel_fid"].astype(int),
                "DN":         -1,
                "area_m2":    0.0,
                "percentage": 0.0,
            },
            geometry=ca_outside.geometry.values, crs=utm,
        )[["parcel_fid", "DN", "area_m2", "percentage", "geometry"]]
        final_gdf = gpd.GeoDataFrame(
            pd.concat([inside_gdf, outside_gdf], ignore_index=True),
            crs=utm,
        )
    else:
        final_gdf = inside_gdf

    _save(final_gdf, final_path, SAVE_SHP)

    gc.collect()
    return {"parcels": len(ca), "features": len(inter_gdf)}

def run_dominant_mapping(results_dir, cadastral_path, raster_path=None):
    # entry point for stage 3 - processes all vector files in results_dir
    cadastral = _load_boundary(cadastral_path)
    vectors   = sorted(f for f in os.listdir(results_dir) if f.endswith("_vector.gpkg"))
    if not vectors:
        log("No vector files found")
        return

    log("Stage 3: Class Mapping")
    print()

    for v in vectors:
        log(f"Processing: {v}")
        with Timer() as t:
            result = _map_one(
                results_dir / v, cadastral, results_dir, raster_path=raster_path
            )

        if result:
            print(f"  Total parcels            : {result['parcels']:,}")
            print(f"  Intersections            : {result['features']:,}")
            print(f"Stage 3 completed in {t.pretty()}")
        else:
            print("  Skipped - already processed")
        print()
        gc.collect()

Writing /content/pipeline/src/class_mapping.py


In [ ]:
# main.py
# runs Stages 1-3: classification, vectorisation, class mapping

%%writefile /content/pipeline/main.py
import sys
import shutil
import time
from pathlib import Path
sys.path.insert(0, "/content/pipeline")
from config import (
    SOURCE_IMAGE, BOUNDARY_FILE,
    RAW_DIR, RESULTS_DIR, PLOT_DIR, BAND_ORDER,
)
from utils import Timer, fmt_time, log
from src.kmeans_classification import run_classification
from src.vectorize_raster      import run_vectorization
from src.class_mapping         import run_dominant_mapping


def _validate():
    # check all required input files exist before starting
    errors = []

    if not SOURCE_IMAGE.exists():
        errors.append(f"Image not found - {SOURCE_IMAGE}")
    else:
        log(f"Input ready: {SOURCE_IMAGE.name}")

    p = Path(BOUNDARY_FILE)
    if p.is_dir():
        shps = list(p.glob("*.shp"))
        if not shps:
            errors.append("No .shp file found in boundary folder")
        else:
            log(f"Boundary folder detected - using: {shps[0].name}")
    elif p.suffix.lower() in (".gpkg", ".shp"):
        if not p.exists():
            errors.append(f"Boundary file not found - {p.name}")
        else:
            log(f"Boundary file ready: {p.name}")
    else:
        errors.append(f"Unsupported boundary format - {p.suffix}")

    if errors:
        print()
        for e in errors:
            print(f"[ERROR] {e}")
        sys.exit(1)

def _prepare():
    # create output dirs and copy source image to data/raw if needed
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    dst = RAW_DIR / SOURCE_IMAGE.name
    if not dst.exists():
        shutil.copy2(SOURCE_IMAGE, dst)
        log(f"Copied {SOURCE_IMAGE.name} to data/raw/")

def main():
    start = time.time()
    print()
    log(f"Pipeline started at {time.strftime('%Y-%m-%d')}")
    print()
    _validate()
    _prepare()
    print()

    image_configs = {SOURCE_IMAGE.name: {"band_order": BAND_ORDER}}

    with Timer() as t1:
        best_k = run_classification(
            RAW_DIR, RESULTS_DIR, PLOT_DIR, image_configs
        )
    print()

    with Timer() as t2:
        run_vectorization(RESULTS_DIR)
    print()

    classified_path = RESULTS_DIR / f"{SOURCE_IMAGE.stem}_classified.tif"
    with Timer() as t3:
        run_dominant_mapping(RESULTS_DIR, BOUNDARY_FILE, raster_path=classified_path)

    log(f"Total runtime: {fmt_time(time.time() - start)}")
    print()

if __name__ == "__main__":
    main()

Writing /content/pipeline/main.py


In [ ]:
# Run main.py
# executes Stages 1-3 (classification, vectorisation, class mapping)
%cd /content/pipeline
!python main.py

/content/pipeline

[INFO] Pipeline started at 2026-05-01

[INFO] Input ready: Gandhinagar.tiff
[INFO] Boundary folder detected - using: gandhinagar_prediction_F.shp
[INFO] Copied Gandhinagar.tiff to data/raw/

[INFO] Stage 1: KMeans Classification

[INFO] Processing: Gandhinagar.tiff
  Bands      : 4 bands [B=3, G=2, R=1, NIR=auto-dead]
  Features   : 15
  best_k = 5 | WCSS = 617,654.75 | Sil = 0.2559
  Output     : Gandhinagar_classified.tif
Stage 1 completed in 0m 39s


[INFO] Stage 2: Raster to Vector Conversion

[INFO] Processing: Gandhinagar_classified.tif
  Polygons : 474,479
  Output   : Gandhinagar_vector.gpkg
Stage 2 completed in 0m 51s


[INFO] Boundary folder detected - using: gandhinagar_prediction_F.shp
[INFO] Stage 3: Class Mapping

[INFO] Processing: Gandhinagar_vector.gpkg
  Parcels in boundary      : 669,359
  Outside raster extent    : 534,856  (labelled DN=-1)
  Parcels inside raster    : 134,503
  Running spatial join...
  Candidate pairs          : 699,242
  Chunks

In [ ]:
# Save pipeline to Google Drive
import shutil
from pathlib import Path

PIPELINE_DIR = Path("/content/pipeline")
DRIVE_DIR    = Path("/content/drive/MyDrive/pipeline_output")

print("Copying pipeline folder to Google Drive...")

if DRIVE_DIR.exists():
    shutil.rmtree(DRIVE_DIR)

shutil.copytree(PIPELINE_DIR, DRIVE_DIR)

print(f"Pipeline saved to: {DRIVE_DIR}")

Copying pipeline folder to Google Drive...
Pipeline saved to: /content/drive/MyDrive/pipeline_output
